# 5. Industry Risk Integration into APRA LGD Model

This notebook demonstrates the integration of industry-level risk analysis outputs into the LGD modelling framework. The industry analysis project produces risk scores, working capital metrics, stress scenarios, and benchmarks for 9 ANZSIC industry divisions, sourced from ABS and RBA public data.

**Integration points:**
1. Industry-sensitive downturn scalars (replacing flat scalars)
2. Industry-adjusted margins of conservatism
3. Recovery haircuts by industry risk level
4. Working capital LGD overlays
5. Enhanced segmentation by industry risk band
6. Industry risk as slotting input (development finance)

**Products affected:** Commercial Cash Flow, Development Finance

**Regulatory alignment:** APRA APS 113 (IRB), APS 112 (Standardised fallback), APS 220 (Credit Risk Management)

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.data_generation import generate_commercial_data, generate_development_data
from src.lgd_calculation import CommercialLGDEngine, DevelopmentLGDEngine, exposure_weighted_average
from src.industry_risk_integration import (
    IndustryRiskLoader, INDUSTRY_NAME_MAP, UNMAPPED_DEFAULTS,
    compute_industry_downturn_scalar, compute_industry_moc_adjustment,
    compute_industry_recovery_haircut, compute_working_capital_lgd_adjustment,
    enrich_loans_with_industry_risk, map_development_type_to_industry,
)
from src.validation import (
    accuracy_metrics, discriminatory_power, conservatism_test,
    calibration_by_segment, compute_psi, generate_validation_report,
    industry_attribution_analysis, compare_models,
)

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:.4f}".format)

print("Setup complete.")

## Core Methodology Update
- All LGD comparisons in this notebook are EAD-weighted for portfolio relevance.
- Discounting assumptions are based on `max(contract_rate_proxy, cost_of_funds_proxy)` in source workout data.
- Downturn LGD starts from product macro drivers, then applies industry-risk overlays (not a flat unexplained scalar).


## 1. Load Industry Analysis Outputs

The industry analysis project produces risk scorecards for 9 ANZSIC industry divisions based on ABS employment, margin, inventory, and demand data combined with structural classification risk scores.

In [ ]:
# Load industry analysis data
INDUSTRY_DATA_PATH = os.path.join(os.getcwd(), '..', 'data', 'exports')
loader = IndustryRiskLoader(INDUSTRY_DATA_PATH)

# Display risk scorecard
scorecard = loader.load_base_risk_scorecard()
print("=== Industry Base Risk Scorecard ===")
print(f"Score range: {scorecard['industry_base_risk_score'].min():.2f} - {scorecard['industry_base_risk_score'].max():.2f}")
print(f"Risk levels: {scorecard['industry_base_risk_level'].value_counts().to_dict()}\n")
scorecard.sort_values("industry_base_risk_score", ascending=False)

In [ ]:
# Visualise risk scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart of risk scores
sc_sorted = scorecard.sort_values("industry_base_risk_score", ascending=True)
colors = sc_sorted["industry_base_risk_level"].map(
    {"Low": "#4CAF50", "Medium": "#FF9800", "Elevated": "#F44336"}
)
axes[0].barh(sc_sorted["industry"], sc_sorted["industry_base_risk_score"], color=colors)
axes[0].axvline(x=2.5, color="gray", linestyle="--", label="Low/Medium boundary")
axes[0].axvline(x=3.0, color="red", linestyle="--", alpha=0.5, label="Medium/Elevated boundary")
axes[0].set_xlabel("Industry Base Risk Score (1-5)")
axes[0].set_title("Industry Risk Scores")
axes[0].legend(fontsize=8)

# Score decomposition
sc_sorted.plot.barh(
    x="industry",
    y=["classification_risk_score", "macro_risk_score"],
    ax=axes[1], color=["#2196F3", "#9C27B0"]
)
axes[1].set_xlabel("Component Score (1-5)")
axes[1].set_title("Risk Score Decomposition (Classification vs Macro)")
axes[1].legend(["Classification (55%)", "Macro (45%)"], fontsize=8)

plt.tight_layout()
plt.show()

## 2. Industry Name Mapping and Data Enrichment

The LGD model uses short industry names (e.g. "Transport") while the industry analysis uses full ANZSIC names (e.g. "Transport, Postal And Warehousing"). The mapping reconciles these. Mining has no match in the industry analysis and receives a conservative Elevated default.

In [ ]:
# Show industry name mapping
mapping_df = pd.DataFrame([
    {"LGD Model Name": k, "ANZSIC Name": v if v else "(No match - conservative default)"}
    for k, v in INDUSTRY_NAME_MAP.items()
])
print("=== Industry Name Mapping ===")
mapping_df

In [ ]:
# Generate synthetic data and enrich with industry risk
com_loans, com_cfs = generate_commercial_data()
dev_loans, dev_cfs = generate_development_data()

# Enrich with compact upstream-parquet industry risk features
com_enriched = enrich_loans_with_industry_risk(com_loans, loader, "commercial")
dev_enriched = enrich_loans_with_industry_risk(dev_loans, loader, "development")

print(f"Commercial loans: {len(com_enriched)} rows, industry risk columns added:")
print([c for c in com_enriched.columns if "industry" in c or "wc_" in c])
print(f"\nDevelopment loans: {len(dev_enriched)} rows, industry risk columns added:")
print([c for c in dev_enriched.columns if "industry" in c or "wc_" in c])

## 3. Industry-Sensitive Downturn Scalars
The baseline model now applies **macro-linked downturn logic by product** (not a flat scalar). Industry risk then adjusts that macro-linked base:
$$	ext{adjusted scalar} = 	ext{macro-linked base scalar} 	imes \left(1 + lpha rac{	ext{risk score} - 2.5}{2.0}ight)$$
This keeps the downturn process transparent: product economics first, industry cyclicality second.


In [ ]:
# Demonstrate industry-adjusted downturn scalars
risk_lookup = loader.get_risk_score_lookup()
base_scalars = {"Property": 1.15, "PPSR - P&E": 1.20, "PPSR - Receivables": 1.20, "PPSR - Mixed": 1.20, "GSR Only": 1.15}

rows = []
for industry, risk_score in sorted(risk_lookup.items(), key=lambda x: -x[1]):
    for sec_type, base in base_scalars.items():
        adjusted = compute_industry_downturn_scalar(risk_score, base)
        rows.append({
            "Industry": industry, "Risk Score": risk_score, "Risk Level": "Elevated" if risk_score > 3.0 else "Medium",
            "Security Type": sec_type, "Base Scalar": base, "Adjusted Scalar": round(adjusted, 4),
            "Change %": round((adjusted / base - 1) * 100, 2)
        })

scalar_df = pd.DataFrame(rows)

# Show summary: one row per industry with Property scalar
summary = scalar_df[scalar_df["Security Type"] == "Property"][
    ["Industry", "Risk Score", "Risk Level", "Base Scalar", "Adjusted Scalar", "Change %"]
]
print("=== Industry-Adjusted Downturn Scalars (Property Security) ===")
summary

In [ ]:
# Visualise: macro-linked base vs industry-adjusted scalars
fig, ax = plt.subplots(figsize=(12, 5))
prop_df = scalar_df[scalar_df["Security Type"] == "Property"].sort_values("Risk Score")
x = np.arange(len(prop_df))
width = 0.35

ax.bar(x - width/2, prop_df["Base Scalar"], width, label="Macro-Linked Base Scalar", color="#90CAF9")
ax.bar(x + width/2, prop_df["Adjusted Scalar"], width, label="Industry-Adjusted Scalar", color="#EF5350")
ax.set_xticks(x)
ax.set_xticklabels(prop_df["Industry"], rotation=45, ha="right")
ax.set_ylabel("Downturn Scalar")
ax.set_title("Macro-Linked Base vs Industry-Adjusted Downturn Scalars (Property Security)")
ax.legend()
ax.axhline(y=1.15, color="gray", linestyle=":", alpha=0.5)

for i, row in enumerate(prop_df.itertuples()):
    ax.annotate(f"{row._7:+.1f}%", (x[i] + width/2, row._6 + 0.005),
                ha="center", fontsize=8, color="red")

plt.tight_layout()
plt.show()


## 4. Commercial LGD with Industry Risk -- Enhanced Pipeline vs Baseline
We run both the baseline (**macro-linked by product, no industry risk**) and enhanced (**macro-linked + industry risk**) pipelines on the same commercial data and compare outcomes.


In [ ]:
# --- BASELINE: Macro-linked downturn (no industry risk overlay) ---
com_engine = CommercialLGDEngine()

# Run baseline on raw data (without industry_risk_score so only product macro drivers apply)
com_baseline_data = com_loans.drop(columns=["industry_risk_score", "industry_risk_level",
                                              "industry_debt_to_ebitda_benchmark",
                                              "industry_icr_benchmark"], errors="ignore")
com_baseline = com_engine.apply_overlays(com_baseline_data)

# --- ENHANCED: Macro-linked + industry-adjusted overlays ---
com_enhanced = com_engine.apply_overlays(com_enriched)

# Compare summary statistics (EAD-weighted)
print("=== Commercial LGD: Baseline vs Enhanced ===
")
compare_cols = ["realised_lgd", "lgd_downturn", "lgd_with_moc", "lgd_final"]
comparison = pd.DataFrame({
    "Metric": compare_cols,
    "Baseline EAD-Weighted": [exposure_weighted_average(com_baseline, c, 'ead') for c in compare_cols],
    "Enhanced EAD-Weighted": [exposure_weighted_average(com_enhanced, c, 'ead') for c in compare_cols],
})
comparison["Diff (pp)"] = (comparison["Enhanced EAD-Weighted"] - comparison["Baseline EAD-Weighted"]) * 100
comparison


In [ ]:
# Waterfall chart: LGD pipeline stages for enhanced model by industry risk band
com_segmented = com_engine.segment_loans(com_enhanced)

rows = []
for band, grp in com_segmented.groupby("industry_risk_band", observed=True):
    rows.append({
        "industry_risk_band": band,
        "realised_lgd": exposure_weighted_average(grp, "realised_lgd", "ead"),
        "lgd_industry_adjusted": exposure_weighted_average(grp, "lgd_industry_adjusted", "ead"),
        "lgd_downturn": exposure_weighted_average(grp, "lgd_downturn", "ead"),
        "lgd_with_moc": exposure_weighted_average(grp, "lgd_with_moc", "ead"),
        "lgd_final": exposure_weighted_average(grp, "lgd_final", "ead"),
        "count": len(grp),
        "total_ead": grp["ead"].sum(),
    })
waterfall_data = pd.DataFrame(rows)

print("=== LGD Pipeline Stages by Industry Risk Band (EAD-Weighted) ===")
waterfall_data


## 5. Development LGD with Industry Risk -- Enhanced Pipeline vs Baseline

In [ ]:
# --- Development: Baseline vs Enhanced ---
dev_engine = DevelopmentLGDEngine()

# Baseline: strip industry columns
dev_baseline_data = dev_loans.drop(columns=["industry", "industry_risk_score",
                                              "industry_risk_level"], errors="ignore")
dev_baseline = dev_engine.apply_overlays(dev_baseline_data)

# Enhanced: with industry risk
dev_enhanced_overlay = dev_engine.apply_overlays(dev_enriched)

# Compare (EAD-weighted)
print("=== Development LGD: Baseline vs Enhanced ===
")
dev_comparison = pd.DataFrame({
    "Metric": compare_cols,
    "Baseline EAD-Weighted": [exposure_weighted_average(dev_baseline, c, 'ead') for c in compare_cols],
    "Enhanced EAD-Weighted": [exposure_weighted_average(dev_enhanced_overlay, c, 'ead') for c in compare_cols],
})
dev_comparison["Diff (pp)"] = (dev_comparison["Enhanced EAD-Weighted"] - dev_comparison["Baseline EAD-Weighted"]) * 100
dev_comparison


In [ ]:
# Slotting comparison: baseline vs enhanced
dev_baseline_slotted = dev_engine.assign_slotting(dev_baseline)
dev_enhanced_slotted = dev_engine.assign_slotting(dev_enhanced_overlay)

print("=== Slotting Distribution: Baseline vs Enhanced ===\n")
slot_comp = pd.DataFrame({
    "Baseline": dev_baseline_slotted["slotting_category"].value_counts().sort_index(),
    "Enhanced": dev_enhanced_slotted["slotting_category"].value_counts().sort_index(),
})
slot_comp["Shift"] = slot_comp["Enhanced"] - slot_comp["Baseline"]
slot_comp

## 6. Industry Segmentation Heatmaps

Cross-tabulation of final LGD by security type and industry risk band reveals which combinations carry the highest loss severity.

In [ ]:
# Heatmap: LGD by security type x industry risk band (EAD-weighted)
com_seg = com_engine.segment_loans(com_enhanced)

heatmap_data = (
    com_seg.groupby(["security_type", "industry_risk_band"], observed=True)
    .apply(lambda g: exposure_weighted_average(g, "lgd_final", "ead"), include_groups=False)
    .unstack(fill_value=np.nan)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Final LGD heatmap
sns.heatmap(heatmap_data, annot=True, fmt=".3f", cmap="YlOrRd", ax=axes[0],
            vmin=0.2, vmax=0.9, linewidths=0.5)
axes[0].set_title("EAD-Weighted Final LGD: Security Type x Industry Risk Band")
axes[0].set_ylabel("Security Type")
axes[0].set_xlabel("Industry Risk Band")

# Count heatmap
count_data = com_seg.groupby(["security_type", "industry_risk_band"], observed=True)[
    "loan_id"
].count().unstack(fill_value=0)
sns.heatmap(count_data, annot=True, fmt="d", cmap="Blues", ax=axes[1], linewidths=0.5)
axes[1].set_title("Loan Count: Security Type x Industry Risk Band")
axes[1].set_ylabel("Security Type")
axes[1].set_xlabel("Industry Risk Band")

plt.tight_layout()
plt.show()


## 7. Stress Testing with Industry Scenarios

The industry analysis provides stress test results for 4 scenarios: demand shock, margin squeeze, employment decline, and rate shock. We use these to model industry-specific downturn LGD under stress.

In [ ]:
# Load stress matrix
stress_matrix = loader.load_stress_matrix()
print(f"Stress scenarios: {stress_matrix['scenario_name'].unique().tolist()}")
print(f"Industries covered: {stress_matrix['industry'].nunique()}\n")

# Show stress impact on LGD
stress_summary = stress_matrix.groupby("scenario_name").agg(
    mean_stress_delta=("stress_delta", "mean"),
    max_stress_delta=("stress_delta", "max"),
    escalation_count=("implied_monitoring_action", lambda x: (x == "Escalate sector review").sum()),
).reset_index()

print("=== Stress Scenario Summary ===")
stress_summary

In [ ]:
# Apply worst-case stress scenario to commercial LGD
# Use stressed industry risk scores (employment decline = worst scenario)
emp_stress = stress_matrix[stress_matrix["scenario_name"] == "Employment decline"].copy()

com_stressed = com_enriched.copy()
stress_lookup = dict(zip(emp_stress["industry"], emp_stress["stressed_industry_risk_score"]))
com_stressed["industry_risk_score"] = com_stressed["industry"].map(stress_lookup).fillna(
    com_stressed["industry_risk_score"]
)

# Re-run overlay engine with stressed risk scores
com_stressed_overlay = com_engine.apply_overlays(com_stressed)

# Compare base vs stressed LGD by industry (EAD-weighted)
rows = []
for ind, grp in com_stressed_overlay.groupby("industry", observed=True):
    base_grp = com_enhanced[com_enhanced["industry"] == ind]
    rows.append({
        "industry": ind,
        "enhanced_lgd": exposure_weighted_average(base_grp, "lgd_final", "ead") if len(base_grp) else np.nan,
        "stressed_lgd": exposure_weighted_average(grp, "lgd_final", "ead"),
        "count": len(grp),
        "total_ead": grp["ead"].sum(),
    })
stress_compare = pd.DataFrame(rows)
stress_compare["stress_uplift_pp"] = (stress_compare["stressed_lgd"] - stress_compare["enhanced_lgd"]) * 100

print("=== Employment Decline Stress Impact on LGD (EAD-Weighted) ===")
stress_compare.sort_values("stress_uplift_pp", ascending=False)


## 8. Working Capital Overlay Impact

Industries with poor working capital profiles (high CCC, slow collections, illiquid inventory) face higher LGD because recoveries during administration are lower.

In [ ]:
# Working capital metrics
wc_metrics = loader.load_working_capital_metrics()
wc_display = wc_metrics[["industry", "cash_conversion_cycle_benchmark_days",
                          "cash_conversion_cycle_stress_days",
                          "working_capital_lgd_overlay_score",
                          "working_capital_lgd_overlay_band",
                          "lgd_primary_driver"]].sort_values(
    "working_capital_lgd_overlay_score", ascending=False
)

print("=== Working Capital LGD Overlay by Industry ===")
print("WC LGD overlay adds 0-1.9pp to LGD for industries with poor WC profiles\n")

# Show the additive adjustment
wc_display["lgd_adjustment_pp"] = compute_working_capital_lgd_adjustment(
    wc_display["working_capital_lgd_overlay_score"].values
) * 100
wc_display

## 9. ESG Sensitivity Overlay

ESG-sensitive sectors (Agriculture, Manufacturing, Construction, Transport, Accommodation) receive enhanced due diligence flags that can inform monitoring intensity for LGD estimates.

In [ ]:
# ESG overlay
esg = loader.load_esg_overlay()
print("=== ESG Sensitivity by Industry ===\n")
esg[["industry", "esg_sensitive_sector", "esg_focus_area", "credit_policy_overlay"]]

In [ ]:
# ESG impact: compare LGD for ESG-sensitive vs non-sensitive sectors
com_enhanced["esg_group"] = com_enhanced["industry_esg_sensitive"].map(
    {True: "ESG Sensitive", False: "Non-Sensitive"}
)
rows = []
for grp_name, grp in com_enhanced.groupby("esg_group", observed=True):
    rows.append({
        "esg_group": grp_name,
        "ead_weighted_realised_lgd": exposure_weighted_average(grp, "realised_lgd", "ead"),
        "ead_weighted_final_lgd": exposure_weighted_average(grp, "lgd_final", "ead"),
        "mean_risk_score": grp["industry_risk_score"].mean(),
        "count": len(grp),
        "total_ead": grp["ead"].sum(),
    })
esg_comp = pd.DataFrame(rows)

print("=== LGD by ESG Sensitivity ===")
esg_comp


## 10. Validation Comparison: Baseline vs Enhanced Model

We run the full validation framework on both models to confirm the industry-enhanced model provides better or equal performance.

In [ ]:
# Prepare comparison DataFrame
com_compare = com_enhanced.copy()
com_compare["lgd_final_baseline"] = com_baseline["lgd_final"].values
com_compare["lgd_final"] = com_enhanced["lgd_final"].values

# Run model comparison
comparison_results = compare_models(
    com_compare,
    actual_col="realised_lgd",
    baseline_col="lgd_final_baseline",
    enhanced_col="lgd_final",
)

print("=== Accuracy Comparison ===")
print(comparison_results["accuracy"].to_string(index=False))
print("\n=== Discriminatory Power ===")
print(comparison_results["discriminatory_power"].to_string(index=False))
print("\n=== Conservatism ===")
print(comparison_results["conservatism"].to_string(index=False))

if "model_shift_psi" in comparison_results:
    psi = comparison_results["model_shift_psi"]
    print(f"\n=== Model Shift PSI: {psi['PSI']:.4f} ({psi['Interpretation']}) ===")

In [ ]:
# Industry attribution analysis
com_seg_enhanced = com_engine.segment_loans(com_enhanced)
attribution = industry_attribution_analysis(
    com_seg_enhanced,
    actual_col="realised_lgd",
    predicted_col="lgd_final",
    industry_col="industry_risk_band",
    security_col="security_type",
)

print("=== Industry Risk Attribution ===")
print(f"R-squared (industry alone):          {attribution.get('r2_industry_alone', 'N/A')}")
print(f"R-squared (security type alone):     {attribution.get('r2_security_alone', 'N/A')}")
print(f"R-squared (combined):                {attribution.get('r2_combined', 'N/A')}")
print(f"Incremental R-squared from industry: {attribution.get('r2_incremental_industry', 'N/A')}")
print("\n=== Calibration by Industry Risk Band ===")
attribution["calibration_by_industry"]

## 11. Concentration Risk Analysis

Portfolio concentration limits from the industry analysis flag sectors where exposure exceeds risk-calibrated caps.

In [ ]:
# Concentration limits
conc = loader.load_concentration_limits()
print("=== Portfolio Concentration Limits ===")
print("Breached sectors have exposure exceeding risk-calibrated caps.\n")

# Highlight breaches
conc_styled = conc.sort_values("utilisation_pct", ascending=False)
conc_styled

## 12. Summary and APRA Compliance Checklist

### Integration Impact Summary
- Industry risk scores (1-5 scale) are integrated into **downturn scalars**, **margins of conservatism**, and **recovery haircuts**
- Commercial LGD pipeline gains industry-sensitive overlays with bounded adjustments (+/- 7-10%)
- Development LGD gains industry risk as a slotting input and overlay adjustment
- Working capital overlay adds 0-1.9pp for industries with poor liquidity profiles
- ESG sensitivity flags enhance monitoring intensity

### APRA Compliance Checklist
| Requirement | Status |
|:---|:---|
| LGD floors (10% std / 15% non-std mortgage) preserved | Floors remain hard constraints, applied after all industry adjustments |
| Supervisory LGD fallback (APS 112) maintained | Senior Secured 35%, Senior Unsecured 45% floors unchanged |
| Downturn LGD reflects severe economic conditions (APS 113) | Industry-specific scalars more defensible than flat scalars |
| MoC reflects data limitations and model uncertainty | Industry-varying MoC appropriate -- data depth varies by sector |
| No double-counting with PD model | LGD adjustment targets recovery/workout channels, not default probability |
| Slotting framework (4 categories) preserved | Industry risk is additive input; Strong/Good/Satisfactory/Weak structure unchanged |
| Conservatism direction (net LGD-increasing) | Industry adjustments calibrated with midpoint below portfolio-weighted average |
| Documentation and audit trail | All formulas, parameters, and calibration rationale documented |